### 第 1 章 强化学习入门


#### 1.1 强化学习概述

强化学习讨论的是智能体（Agent）如何在复杂、不确定的环境中，通过与环境的交互来最大化累积奖励。

![强化学习交互示意图](../images/image.png)


##### 1.1.1 强化学习与监督学习的区别

**监督学习**：从标注数据中学习输入到输出的映射 $f: \mathcal{X} \rightarrow \mathcal{Y}$

**强化学习**：智能体通过与环境交互，学习最优策略以最大化累积奖励

| 特性 | 监督学习 | 强化学习 |
|------|---------|--------|
| 数据来源 | 静态数据集 | 与环境交互 |
| 反馈形式 | 标签（正确答案） | 奖励信号（标量） |
| 学习目标 | 最小化预测误差 | 最大化累积奖励 |
| 决策序列 | 独立同分布 | 时序相关 |


##### 1.1.2 强化学习的核心要素

强化学习问题通常用**马尔可夫决策过程**（MDP）的五元组 $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$ 来描述：

---

**$\mathcal{S}$：状态空间**（State Space）

状态是对世界的完整描述，不会隐藏信息。观测（Observation）是对状态的部分描述，可能会遗漏一些信息。

- 当智能体能够完全观测到环境的所有状态时，称为**完全可观测**（Fully Observable）
- 否则称为**部分可观测**（Partially Observable），对应 POMDP

---

**$\mathcal{A}$：动作空间**（Action Space）

在给定环境中，智能体可以执行的有效动作的集合。

- **离散动作空间**：有限个动作可选（如上下左右）
- **连续动作空间**：动作是连续值（如方向盘角度、油门力度）

---

**$\mathcal{P}$：状态转移概率**（Transition Probability）

$$P(s'|s,a) = P(S_{t+1}=s' \mid S_t=s, A_t=a)$$

描述在状态 $s$ 下执行动作 $a$ 后，转移到状态 $s'$ 的概率。

---

**$\mathcal{R}$：奖励函数**（Reward Function）

$$R(s,a) = \mathbb{E}[R_{t+1} \mid S_t=s, A_t=a]$$

描述在状态 $s$ 执行动作 $a$ 后，**平均**能获得多少奖励。

> **即时奖励 vs 奖励函数**：
> - **即时奖励** $R_{t+1}$：随机变量，是环境在 $t+1$ 时刻实际返回的奖励值
> - **奖励函数** $R(s,a)$：确定性函数，是即时奖励的期望值
> - 随机性来源：同一个 $(s,a)$ 可能转移到不同的 $s'$，对应不同的奖励

**严格定义**：

MDP 的环境动态完全由**联合概率函数** $p(s', r \mid s, a)$ 描述，它表示在状态 $s$ 执行动作 $a$ 后，**同时**转移到状态 $s'$ 并获得奖励 $r$ 的概率。所有其他量都可以从它推导出来：

**1. 状态转移概率**（对 $r$ 边缘化）：

$$p(s' \mid s, a) = \sum_{r \in \mathcal{R}} p(s', r \mid s, a)$$

**2. 奖励函数**（对所有 $s', r$ 求期望）：

$$R(s,a) = \mathbb{E}[R_{t+1} \mid S_t=s, A_t=a] = \sum_{r \in \mathcal{R}} r \sum_{s' \in \mathcal{S}} p(s', r \mid s, a)$$

**3. 条件期望奖励**（给定下一状态 $s'$ 时的期望奖励）：

$$r(s,a,s') = \mathbb{E}[R_{t+1} \mid S_t=s, A_t=a, S_{t+1}=s'] = \sum_{r \in \mathcal{R}} r \cdot \frac{p(s', r \mid s, a)}{p(s' \mid s, a)}$$

> 推导：由条件概率公式 $P(r \mid s, a, s') = \frac{p(s', r \mid s, a)}{p(s' \mid s, a)}$，然后对 $r$ 求期望。

**等价关系**：奖励函数可以通过条件期望奖励和状态转移概率表示：

$$R(s,a) = \sum_{s' \in \mathcal{S}} p(s' \mid s, a) \cdot r(s,a,s')$$

> 推导：这是全期望公式的应用——先按 $s'$ 分组求条件期望，再加权平均。

**三者的关系**：

```
            p(s', r | s, a)          ← 最基本的环境动态模型
               /        \
  对 r 求和 ↓             ↓ 对 s' 求和后求期望
      p(s'|s,a)         R(s,a)      ← 状态转移概率 / 奖励函数
          ↓
   r(s,a,s')                        ← 条件期望奖励
```

---

**$\gamma$：折扣因子**（Discount Factor），$\gamma \in [0,1]$

用于衡量未来奖励相对于当前奖励的重要性。


##### 1.1.3 智能体的核心组成部分

---

**策略（Policy）**

策略描述了智能体在给定状态下如何选择动作，分为两种类型：

- **随机性策略** $\pi(a|s) = P(A_t=a \mid S_t=s)$：输入一个状态，输出动作的概率分布，然后对该分布进行采样得到实际动作。例如：以 0.7 的概率往左，0.3 的概率往右。

- **确定性策略** $a = \pi(s)$：在每个状态下直接输出一个确定的动作。

---

**价值函数（Value Function）**

价值函数用于评估一个状态（或状态-动作对）的好坏，是对未来累积奖励的预测。

**状态价值函数** $V_\pi(s)$：从状态 $s$ 出发，按策略 $\pi$ 行动，期望获得的累积回报

$$V_{\pi}(s) \doteq \mathbb{E}_{\pi}\left[ G_t \mid S_t=s\right] = \mathbb{E}_{\pi}\left[ \sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t=s\right]$$

**状态-动作价值函数** $Q_\pi(s,a)$：在状态 $s$ 下执行动作 $a$，之后按策略 $\pi$ 行动，期望获得的累积回报

$$Q_{\pi}(s, a) \doteq \mathbb{E}_{\pi}\left[ G_t \mid S_t=s, A_t =a\right] = \mathbb{E}_{\pi}\left[ \sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t=s, A_t = a\right]$$

---

**模型（Model）**

模型描述智能体对环境动态的理解，由两部分构成：

- **状态转移概率**：预测下一个状态
  $$P(s'|s,a) = P(S_{t+1} = s' \mid S_t=s, A_t=a)$$

- **奖励函数**：预测即时奖励
  $$R(s,a) = \mathbb{E}\left[ R_{t+1} \mid S_t=s, A_t=a\right]$$

> **注意**：并非所有强化学习方法都需要模型。根据是否使用模型，RL 方法分为：
> - **Model-based**：学习或利用环境模型（如 Dyna, MBPO）
> - **Model-free**：不学习模型，直接从交互中学习策略或价值函数（如 Q-learning, PPO）

有了策略、价值函数和模型三个组成部分，就形成了一个完整的马尔可夫决策过程。
